In [0]:
catalogs_prefix = 'moustache_markt'
catalogs_list = spark.sql(f"show  catalogs like '{catalogs_prefix}*'")
display(catalogs_list)


for i in catalogs_list.collect():
    display(spark.sql(f"DESCRIBE CATALOG EXTENDED {i[0]}"))

# comments should be more descriptive, the more the better


In [0]:
source_catalog = catalogs_prefix + '_sources'

In [0]:
%sql

show schemas in moustache_markt_sources

In [0]:
%sql 
show volumes in moustache_markt_sources.landing

-- change into collect()

In [0]:
from time import time

start = time()

print("hello")
finish = time()

print(f"Elapsed time: {finish-start} seconds")


In [0]:
start = time()

from collections import Counter

extensions = Counter()

source_systems = ['crm', 'ecom', 'pos', 'erp']

def list_files(path):
    for entry in dbutils.fs.ls(path):
        if entry.isDir():
            list_files(entry.path)          # recurse into subfolders
        else:
            extensions[pathlib_ext(entry.name)] += 1

def pathlib_ext(name):
    return "." + name.split(".")[-1] if "." in name else "(no extension)"

for source_systems_i in source_systems:
    list_files(f"/Volumes/moustache_markt_sources/landing/{source_systems_i}")

for ext, count in extensions.most_common():
    print(f"{ext}: {count} file(s)")

finish = time()

print(f"Elapsed time: {finish-start} seconds")

In [0]:
start = time()

from collections import Counter
from concurrent.futures import ThreadPoolExecutor

source_systems = ['crm', 'ecom', 'pos', 'erp']
roots = [f"/Volumes/moustache_markt_sources/landing/{s}" for s in source_systems]

def pathlib_ext(name):
      return "." + name.split(".")[-1] if "." in name else "(no extension)"

def scan(roots, workers=32):
      extensions = Counter()
      level = list(roots)
      with ThreadPoolExecutor(max_workers=workers) as pool:
          while level:
              next_level = []
              for entries in pool.map(dbutils.fs.ls, level):
                  for e in entries:
                      if e.isDir():
                          next_level.append(e.path)
                      else:
                          extensions[pathlib_ext(e.name)] += 1
              level = next_level
      return extensions

extensions = scan(roots)
for ext, count in extensions.most_common():
      print(f"{ext}: {count} file(s)")

#   Liczbę wątków (workers) możesz podnieść do 64. Jeśli pojawią się błędy throttlingu, zmniejsz ją.

In [0]:
start = time()


from pyspark.sql import functions as F

df = (spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .load(roots)
        .select("path"))

(df.withColumn("name", F.element_at(F.split("path", "/"), -1))
     .withColumn("ext", F.regexp_extract("name", r"(\.[^.]+)$", 1))
     .withColumn("ext", F.when(F.col("ext") == "", "(no extension)").otherwise(F.col("ext")))
     .groupBy("ext").count()
     .orderBy(F.desc("count"))
     .show(100, truncate=False))
  

finish = time()

print(f"Elapsed time: {finish-start} seconds")

# Haczyk: Spark pomija pliki i foldery zaczynające się od _ albo . (np. _SUCCESS, .crc). Jeśli takie
# pliki też chcesz liczyć, wybierz opcję 1.


In [0]:
# Na serverlessie opcja 1 powinna działać. dbutils.fs.ls na Volumes jest tam obsługiwane, a wątki to
#   zwykły Python. Nie mam jednak pewności, czy wywołania dbutils z wielu wątków zachowują się na
#   serverlessie tak samo jak na klasycznym klastrze.

#   Bezpieczniej użyć os.scandir. Volumes są dostępne pod ścieżką /Volumes/... jak zwykły system plików,
#   także na serverlessie. Wtedy w ogóle nie korzystasz z dbutils, a scandir od razu mówi, co jest
#   folderem, bez dodatkowych zapytań:

import os
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

source_systems = ['crm', 'ecom', 'pos', 'erp']
roots = [f"/Volumes/moustache_markt_sources/landing/{s}" for s in source_systems]

def pathlib_ext(name):
    return "." + name.split(".")[-1] if "." in name else "(no extension)"

def ls(path):
    with os.scandir(path) as it:
        return [(e.path, e.name, e.is_dir()) for e in it]

def scan(roots, workers=32):
    extensions = Counter()
    level = list(roots)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        while level:
            next_level = []
            for entries in pool.map(ls, level):
                for path, name, is_dir in entries:
                    if is_dir:
                        next_level.append(path)
                    else:
                        extensions[pathlib_ext(name)] += 1
            level = next_level
    return extensions

extensions = scan(roots)
for ext, count in extensions.most_common():
    print(f"{ext}: {count} file(s)")

#   Na co uważać na serverlessie:

#   - Ścieżki: dbutils.fs.ls zwraca ścieżki w formacie dbfs:/Volumes/..., a os potrzebuje /Volumes/....
#     Nie mieszaj tych dwóch formatów. W kodzie wyżej wszystko idzie przez os, więc problem nie występuje.
#   - Liczba wątków: driver na serverlessie jest mały, ale te wątki prawie cały czas czekają na odpowiedź,
#     więc 16–32 powinno być w porządku. Jeśli zaczną się timeouty, zmniejsz workers.
#   - Opcja 2 (binaryFile) też działa na serverlessie, bo to zwykły odczyt przez DataFrame. Przy bardzo
#     dużej liczbie plików może być najszybsza, bo listowanie odbywa się równolegle po stronie Sparka, a
#     nie na driverze.
#   - Nie działa: to, co wymaga spark._jvm albo sc._jsc, np. bezpośredni dostęp do Hadoop FileSystem. Na
#     serverlessie nie ma dostępu do JVM.

#   Jeśli to dalej będzie wolne, sprawdź, ile masz folderów, a ile plików. Przy setkach tysięcy plików
#   przejdź na opcję 2 albo na Auto Loader.

In [0]:
dbutils.fs.ls('/Volumes/moustache_markt_sources/landing/erp')

In [0]:
# variables

FORMATS = {"erp": "parquet", "crm": "jsonl", "ecom": "json_array", "pos": "csv"}

In [0]:
# 1. Pobiera listę volume'ów z information_schema.
#   2. Listuje subfoldery pierwszego poziomu w każdym volume.
#   3. Skanuje równolegle każdy subfolder i zbiera unikalne rozszerzenia.

#   Wszystko działa przez os.scandir, więc nadaje się na serverless.

import os
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor

CATALOG = "moustache_markt_sources"

def pathlib_ext(name):
    return "." + name.split(".")[-1] if "." in name else "(no extension)"

def ls(path):
    with os.scandir(path) as it:
        return [(e.path, e.name, e.is_dir()) for e in it]

# 1. Lista volume'ów w katalogu
volumes = spark.sql(f"""
    SELECT volume_schema, volume_name
    FROM {CATALOG}.information_schema.volumes
    ORDER BY volume_schema, volume_name
""")
display(volumes)

samples = {}   # (volume, folder, ext) -> ścieżka pierwszego pliku (alfabetycznie)

# 2. Subfoldery pierwszego poziomu w każdym volume (np. pos/receipt_line)
stats = defaultdict(Counter)   # (volume, folder) -> Counter rozszerzeń
tasks = []                     # ((volume, folder), ścieżka)
for v in volumes.collect():
    root = f"/Volumes/{CATALOG}/{v.volume_schema}/{v.volume_name}"
    for path, name, is_dir in ls(root):
        if is_dir:
            key = (v.volume_name, name)
            stats[key]  # puste foldery też mają się pokazać w wyniku
            tasks.append((key, path))
        else:
            stats[(v.volume_name, "(root)")][pathlib_ext(name)] += 1

# 3. Rekurencyjny skan każdego subfolderu, równolegle, poziom po poziomie
def scan(tasks, workers=32):
    level = tasks
    with ThreadPoolExecutor(max_workers=workers) as pool:
        while level:
            next_level = []
            results = pool.map(ls, [path for _, path in level])
            for (key, _), entries in zip(level, results):
                for path, name, is_dir in entries:
                    if is_dir:
                        next_level.append((key, path))
                    else:
                        stats[key][pathlib_ext(name)] += 1
            level = next_level

scan(tasks)

# Wynik: unikalne rozszerzenia per volume/folder
rows = [
    (vol, folder, sorted(c), dict(c), sum(c.values()))
    for (vol, folder), c in sorted(stats.items())
]
result = spark.createDataFrame(
    rows,
    "volume string, folder string, extensions array<string>, "
    "ext_counts map<string,bigint>, file_count bigint",
)
display(result)

#   Wynik wygląda mniej więcej tak:

#   ┌────────┬──────────────┬───────────────────┬─────────────────────────────┬────────────┐
#   │ volume │    folder    │    extensions     │         ext_counts          │ file_count │
#   ├────────┼──────────────┼───────────────────┼─────────────────────────────┼────────────┤
#   │ erp    │ category     │ [.csv]            │ {.csv → 12}                 │ 12         │
#   ├────────┼──────────────┼───────────────────┼─────────────────────────────┼────────────┤
#   │ pos    │ receipt_line │ [.json, .parquet] │ {.json → 3, .parquet → 480} │ 483        │
#   └────────┴──────────────┴───────────────────┴─────────────────────────────┴────────────┘

#   Kilka uwag:

#   - Pliki leżące bezpośrednio w volume, a nie w subfolderze, trafiają do wiersza (root).
#   - Puste subfoldery też się pokazują, z pustą listą rozszerzeń. Dzięki temu widać, że folder istnieje.
#   - Jeśli w katalogu są inne schematy niż landing i chcesz brać pod uwagę tylko landing, dopisz do
#     zapytania WHERE volume_schema = 'landing'. Jeśli dwa schematy mają volume o tej samej nazwie, dodaj
#     schemat do klucza, inaczej ich statystyki się zsumują.
#   - Kolumna ext_counts to dodatek. Jeśli potrzebujesz tylko unikalnych rozszerzeń, wystarczy kolumna
#     extensions.

In [0]:
# Tak. Najprościej zapamiętać przykładowy plik już podczas skanu, a potem otworzyć go funkcją podglądu.
#   Przykład zapisuję dla każdej pary folder + rozszerzenie, bo w jednym folderze mogą leżeć np. .json i
#   .parquet.

#   1. Zapamiętywanie przykładowego pliku podczas skanu

#   Dodaj ten fragment przed krokiem 2:



def add_file(key, path, name):
    ext = pathlib_ext(name)
    stats[key][ext] += 1
    skey = (*key, ext)
    if skey not in samples or path < samples[skey]:
        samples[skey] = path

#   Następnie w dwóch miejscach podmień linię zliczającą plik na wywołanie add_file.

# W kroku 2 zamień
# stats[(v.volume_name, "(root)")][pathlib_ext(name)] += 1
# na:
# add_file((v.volume_name, "(root)"), path, name)

# W funkcji scan zamień
# stats[key][pathlib_ext(name)] += 1
# na:
# add_file(key, path, name)

# "Pierwszy" plik oznacza tu pierwszy alfabetycznie. scandir zwraca pliki w przypadkowej kolejności, a
# tak przy każdym uruchomieniu dostaniesz ten sam plik.

# 2. Podgląd zawartości

import gzip

def peek(path, n_bytes=2000, n_rows=5):
    name = os.path.basename(path)
    ext = pathlib_ext(name).lower()
    print(f"\n=== {path}")

    # formaty binarne: czyta je Spark i pokazuje kilka wierszy
    if ext in (".parquet", ".orc", ".avro"):
        df = spark.read.format(ext[1:]).load(path)
        df.printSchema()
        display(df.limit(n_rows))
        return

    # pliki skompresowane gzipem: rozpakowuje w locie
    opener = gzip.open if ext == ".gz" else open
    with opener(path, "rb") as f:
        raw = f.read(n_bytes)
    print(raw.decode("utf-8", errors="replace"))

#   Pliki tekstowe (CSV, JSON, TXT, XML) są pokazywane jako surowy tekst. Celowo nie parsuję ich Sparkiem,
#   bo w landing zone chcesz zobaczyć, co faktycznie jest w pliku: separator, nagłówek, czy JSON jest
#   wielolinijkowy, czy plik ma BOM.

#   3. Użycie

#   Podgląd wszystkiego, po jednym pliku na folder i rozszerzenie:

for (vol, folder, ext), path in sorted(samples.items()):
    peek(path)

# Albo podgląd jednego wybranego folderu:

# for (vol, folder, ext), path in samples.items():
#     if (vol, folder) == ("pos", "receipt_line"):
#         peek(path)

# # Ścieżkę przykładowego pliku możesz też dodać jako kolumnę w tabeli wynikowej z poprzedniego kroku:

# rows = [
#     (vol, folder, sorted(c), dict(c), sum(c.values()),
#     {ext: samples[(vol, folder, ext)] for ext in c})
#     for (vol, folder), c in sorted(stats.items())
# ]
# result = spark.createDataFrame(
#     rows,
#     "volume string, folder string, extensions array<string>, "
#     "ext_counts map<string,bigint>, file_count bigint, sample_files map<string,string>",
# )
# display(result)

# Jeśli folderów jest dużo, pętla po wszystkich wyświetli sporo outputu w jednej komórce. Wtedy lepiej
# podglądać po jednym volume na raz.

In [0]:
import json
import os
import sys

import pandas as pd
from pyspark.sql import functions as F

# Repo = folder nad testing/. Dopisujemy go do sys.path, żeby zadziałało `import uc_bootstrap`.
REPO = os.path.dirname(os.getcwd())
if REPO not in sys.path:
    sys.path.insert(0, REPO)
    
from uc_bootstrap import load_config

ENV = "dev"
config = load_config()
SOURCES = config.sources_catalog       # moustache_markt_sources - pliki od systemów źródłowych
CATALOG = config.env_catalog(ENV)      # moustache_markt_dev     - nasza platforma (bronze, silver, gold)
LANDING = config.landing_path          # /Volumes/moustache_markt_sources/landing

# Pierwszy dzień z danych dziennych (01_daily). Tylko w nich jest type corruption, więc testy
# parsowania mają sens przede wszystkim od tej daty.
DAILY_FROM = "2026-06-01"

# Format pliku per system - potrzebny przy czytaniu nagłówków i schematów (krok 5).
FORMATS = {"erp": "parquet", "crm": "jsonl", "ecom": "json_array", "pos": "csv"}

print(f"sources: {SOURCES}\nenv:     {CATALOG}\nlanding: {LANDING}\nsystems: {config.source_systems}")

In [0]:
# Path - an object-oriented file path; works the same on Windows, Linux and in the Databricks Workspace.
from pathlib import Path
DEFAULT_CONFIG_PATH = Path.cwd() / "config" / "platform.yml"
DEFAULT_CONFIG_PATH

In [0]:
config

In [0]:

CATALOG_NAME = 'moustache_markt'
